# Notebook 16 — Potency-Regime Mixture of Experts

**Question:** Can low/high potency specialists reduce regression toward the mean beyond the frozen CheMeleon + screening ensemble?

Runs four alternatives: hard-routing experts, probability-weighted experts, probability-weighted **residual** experts, and a non-gated residual control. The gate uses **only molecular features and model predictions**. True potency class is used only during training and for retrospective metrics, never routing at prediction time.

All corrections are trained on the fully cross-fitted inner-fold predictions produced by Notebook 11, and selected by scaffold-grouped meta-CV. Outermost fold labels are not exposed until evaluation. For source-model-free reproducibility, **no CheMeleon GPU retraining** is required.

**Limits:** This is a second-stage study based on saved OOF base-model outputs; inner-meta training predictions have an overlapping crossfit history and are not a replacement for rebuilding a fully nested end-to-end training system. Treat improvements as internal validation and check multiple seeds before making publication claims.

## 0. Install packages
Run once in Colab. Python ≥3.10 recommended. A GPU is optional for the neural or contrastive experiment; the pretrained CheMeleon model is **not retrained**.

In [ ]:
%pip -q install "numpy>=1.26" "pandas>=2.0" "rdkit>=2023.09" "scikit-learn>=1.4" "scipy>=1.11" "matplotlib>=3.8" "torch>=2.1"

## 1. Configure and upload Notebook 11 results
Download `cyp_notebook_11_results.zip` from the **real five-fold Notebook 11 run**, upload it to Colab, and set its path. Use a Google Drive `PERSISTENT_ROOT` to make interrupted executions restartable. Run `FAST_MODE=True` first; use `False` to produce research-scale tables.

In [ ]:
from pathlib import Path
import os, random
SEED = 2026
FAST_MODE = True  # first run uses outer fold 0; change to False for all five
FOLDS_TO_RUN = [0] if FAST_MODE else [0,1,2,3,4]
FP_BITS = 256  # choose 512 for final full run if CPU budget permits
NB11_RESULTS_ZIP = '/content/cyp_notebook_11_results.zip' if Path('/content').exists() else '/mnt/data/cyp_notebook_11_results.zip'
PERSISTENT_ROOT = None  # e.g. '/content/drive/MyDrive/CYP/PotencyRegime/16'
EXPERIMENT_TAG = 'regime_v1'  # increment after code/method changes to avoid stale caches
OUTPUT = (Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')) / 'cyp_notebook_16_results'
OUTPUT.mkdir(parents=True,exist_ok=True)
assert Path(NB11_RESULTS_ZIP).is_file(), 'Upload the REAL full five-fold Notebook 11 results ZIP and edit NB11_RESULTS_ZIP'
print('Notebook 16', 'FAST' if FAST_MODE else 'FIVE-FOLD', 'cache directory:', OUTPUT)


## 2. Load and audit the saved first-stage OOF predictions
The following standalone utilities parse the archive, confirm the baseline is the fixed 50:50 CheMeleon/screening blend, reconstruct inner-training CI bounds from the corresponding training-compound records, and **assert no outer held-out molecule or scaffold appears in the fitting data**.

In [ ]:
"""Common data handling and leakage-safe utilities embedded into standalone CYP Colab notebooks.
Input is the complete five-fold Notebook 11 results archive. No outer-fold targets enter fitting.
"""
import io, json, zipfile, hashlib, warnings, random, os, sys, math, time
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from scipy.optimize import minimize
ISOS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
BASE_NAME='CheMeleon_finetuned + ScreenStack_LGBM'
CHEM_NAME='CheMeleon_finetuned'
SCREEN_NAME='ScreenStack_LGBM'
DESCRIPTOR_FNS=[('MolWt',Descriptors.MolWt),('MolLogP',Descriptors.MolLogP),('TPSA',Descriptors.TPSA),('NumHDonors',Descriptors.NumHDonors),('NumHAcceptors',Descriptors.NumHAcceptors),('NumRotatableBonds',Descriptors.NumRotatableBonds),('RingCount',Descriptors.RingCount),('FractionCSP3',Descriptors.FractionCSP3),('HeavyAtomCount',Descriptors.HeavyAtomCount),('NumAromaticRings',Descriptors.NumAromaticRings),('NumAliphaticRings',Descriptors.NumAliphaticRings),('NumHeteroatoms',Descriptors.NumHeteroatoms)]

def zread(z,leaf):
    paths=[n for n in z.namelist() if n == leaf or n.endswith('/'+leaf)]
    if len(paths)!=1:raise ValueError(f'{leaf}: expected exactly one archive member, found {paths}')
    return z.read(paths[0])

def load_archive(path,allow_incomplete=False):
    with zipfile.ZipFile(path) as z:
        outer=pd.read_csv(io.BytesIO(zread(z,'nested_outer_predictions.csv')))
        inner=pd.read_csv(io.BytesIO(zread(z,'inner_oof_predictions_for_selection.csv')))
        manifest=pd.read_csv(io.BytesIO(zread(z,'fold_manifest.csv')))
        metadata=json.loads(zread(z,'run_metadata.json'))
    if not allow_incomplete:
        assert metadata.get('scientifically_valid_full_model_run') and metadata.get('complete_five_fold_oof'), 'Need full scientifically valid Notebook 11 output, not proxy/FAST'
    assert {'canonical_smiles','scaffold_group','fold'}<=set(manifest.columns)
    assert manifest.canonical_smiles.is_unique and manifest.scaffold_group.notna().all()
    assert manifest.groupby('scaffold_group').fold.nunique().max()==1
    o=outer[outer.subset.isin([BASE_NAME,CHEM_NAME,SCREEN_NAME])].copy()
    assert set(o.subset.unique())=={BASE_NAME,CHEM_NAME,SCREEN_NAME},'Missing component predictions in Notebook 11 archive'
    assert o[['canonical_smiles','isoform','fold','subset']].duplicated().sum()==0
    names=['y_true','ci_low','ci_high','scaffold_group']
    baseline=o[o.subset==BASE_NAME][['canonical_smiles','isoform','fold','y_true','y_pred','ci_low','ci_high','scaffold_group']].copy().rename(columns={'y_pred':'base'})
    c=o[o.subset==CHEM_NAME][['canonical_smiles','isoform','fold','y_pred']].rename(columns={'y_pred':'chemeleon'})
    s=o[o.subset==SCREEN_NAME][['canonical_smiles','isoform','fold','y_pred']].rename(columns={'y_pred':'screenstack'})
    outer_data=baseline.merge(c,on=['canonical_smiles','isoform','fold'],validate='one_to_one').merge(s,on=['canonical_smiles','isoform','fold'],validate='one_to_one')
    # For this *fixed* candidate baseline use explicitly 50:50 rather than Notebook11's selected pair label.
    assert np.allclose(outer_data.base,0.5*(outer_data.chemeleon+outer_data.screenstack),atol=1e-6)
    assert outer_data.isna().sum().get('y_true',0)==0
    assert set(outer_data.isoform)==set(ISOS)
    inner=inner.copy()
    assert not inner.duplicated(['fold','canonical_smiles','isoform']).any()
    inner['base']=0.5*(inner[CHEM_NAME]+inner[SCREEN_NAME])
    inner['chemeleon']=inner[CHEM_NAME]
    inner['screenstack']=inner[SCREEN_NAME]
    # The outer-prediction file contains CI for every compound *in its own outer fold*.
    # Using that label metadata for molecules INSIDE the current training partition is valid.
    reference=outer_data[['canonical_smiles','isoform','y_true','ci_low','ci_high']]
    inner=inner.merge(reference,on=['canonical_smiles','isoform'],suffixes=('','_outer'),validate='many_to_one')
    assert np.allclose(inner.y_true,inner.y_true_outer,atol=1e-6)
    inner=inner.drop(columns=['y_true_outer'])
    inner=inner.merge(manifest[['canonical_smiles','scaffold_group','fold']].rename(columns={'fold':'true_outer_fold'}),on='canonical_smiles',validate='many_to_one')
    assert (inner.fold!=inner.true_outer_fold).all(),'Inner meta-training contains an outer held-out compound'
    assert (outer_data.groupby('canonical_smiles').fold.nunique()==1).all()
    for fold in sorted(outer_data.fold.unique()):
        train=inner[inner.fold==fold]; held=outer_data[outer_data.fold==fold]
        assert not set(train.canonical_smiles)&set(held.canonical_smiles)
        assert not set(train.scaffold_group)&set(held.scaffold_group)
    assert np.isfinite(outer_data[['base','chemeleon','screenstack','y_true']].to_numpy(float)).all()
    assert np.isfinite(inner[['base','chemeleon','screenstack','y_true']].to_numpy(float)).all()
    return outer_data,inner,manifest,metadata

def fingerprint_features(smiles_list,bits=512):
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=bits)
    X=np.empty((len(smiles_list),bits+len(DESCRIPTOR_FNS)),dtype=np.float32)
    fps=[]
    for i,s in enumerate(smiles_list):
        m=Chem.MolFromSmiles(s)
        if m is None:raise ValueError(f'Invalid SMILES: {s!r}')
        fp=gen.GetFingerprint(m);fps.append(fp)
        arr=np.zeros(bits,dtype=np.int8);DataStructs.ConvertToNumpyArray(fp,arr)
        X[i,:bits]=arr
        for j,(_,fn) in enumerate(DESCRIPTOR_FNS):
            try:v=fn(m);X[i,bits+j]=float(v) if np.isfinite(v) else 0.
            except Exception:X[i,bits+j]=0.
    return X,fps

def feature_table(outer,inner,bits=512):
    all_smiles=sorted(set(outer.canonical_smiles)|set(inner.canonical_smiles))
    arr,fps=fingerprint_features(all_smiles,bits)
    index={s:k for k,s in enumerate(all_smiles)}
    return {'x':arr,'fps':fps,'smiles':all_smiles,'index':index,'bits':bits}

def make_x(frame,features,include_model=True):
    ids=np.fromiter((features['index'][s] for s in frame.canonical_smiles),dtype=int,count=len(frame))
    desc=features['x'][ids]
    if include_model:
        additional=np.stack([frame.base.to_numpy(float),frame.chemeleon.to_numpy(float),frame.screenstack.to_numpy(float),
                             (frame.chemeleon-frame.screenstack).to_numpy(float),
                             np.asarray([ISOS.index(k) for k in frame.isoform],float)/3],axis=1).astype(np.float32)
        return np.column_stack([desc,additional])
    return desc

def metric_stats(frame,pred,original=None):
    local=frame.copy();local['p']=np.asarray(pred,float)
    local['err']=(local.y_true-local.p).abs()
    lo=local.ci_low.to_numpy(float);hi=local.ci_high.to_numpy(float);p=local.p.to_numpy(float)
    valid=np.isfinite(lo)&np.isfinite(hi)&(lo<=hi)
    dist=np.maximum(lo-p,0)+np.maximum(p-hi,0)
    local['excess']=np.where(valid,dist,np.nan)
    out={'macro_mae':float(local.groupby('isoform').err.mean().reindex(ISOS).mean()),
         'macro_interval_excess':float(local.groupby('isoform').excess.mean().reindex(ISOS).mean()),
         'n':len(local)}
    if original is not None:
        out['mae_improvement']=float(metric_stats(frame,original)['macro_mae']-out['macro_mae'])
    return out

def safe_auc(y,p):
    return float(roc_auc_score(y,p)) if len(set(np.asarray(y,dtype=int)))==2 else np.nan

def sample_bootstrap(all_preds,baseline_col='base',B=300,seed=2026):
    p=all_preds.copy(); p['benefit']=(p.y_true-p[baseline_col]).abs()-(p.y_true-p.y_pred).abs()
    ss=p.groupby(['scaffold_group','isoform']).benefit.agg(['sum','count']).reset_index()
    sums=ss.pivot(index='scaffold_group',columns='isoform',values='sum').reindex(columns=ISOS).fillna(0).to_numpy()
    cnt=ss.pivot(index='scaffold_group',columns='isoform',values='count').reindex(columns=ISOS).fillna(0).to_numpy()
    rng=np.random.default_rng(seed);v=[]
    for _ in range(B):
        ind=rng.integers(0,len(sums),len(sums));nr=sums[ind].sum(0);dn=cnt[ind].sum(0)
        if (dn>0).all():v.append(float(np.mean(nr/dn)))
    return {'gain':float(p.groupby('isoform').benefit.mean().reindex(ISOS).mean()),
            'low':float(np.quantile(v,.025)) if len(v) else np.nan,
            'high':float(np.quantile(v,.975)) if len(v) else np.nan,'reps':len(v)}

def wrap_results(output,archive_filename,files):
    import zipfile
    with zipfile.ZipFile(archive_filename,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for fn in files:
            path=Path(output)/fn
            if path.exists():z.write(path,arcname='cyp_regime_results/'+fn)
    print('Wrote',archive_filename)

def guarded_folds(train,held):
    assert not set(train.canonical_smiles)&set(held.canonical_smiles)
    assert not set(train.scaffold_group)&set(held.scaffold_group)

def save_csv(df,path):
    # Normalize floating precision across fresh and cached executions.
    df.to_csv(path,index=False,float_format='%.10g')

def get_root():
    return Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')


In [ ]:
OUTER, INNER, MANIFEST, META11 = load_archive(NB11_RESULTS_ZIP)
FEATURES = feature_table(OUTER, INNER, bits=FP_BITS)
print("Outer observations:", len(OUTER), "Inner meta observations:", len(INNER), "Features:", FEATURES["x"].shape)
print("Source scientifically valid:", META11.get("scientifically_valid_full_model_run"))
assert OUTER.groupby("canonical_smiles").fold.nunique().max()==1

## 3. Fold-safe modeling and ablation
**No outer-fold potency labels are used for fitting, selecting a correction, calibrating a gate, or mining training pairs.** The methods cache each completed fold. Expected outputs include a chosen-method audit, per-CYP performance, and low/high-potency diagnostics.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge

METHODS=['HardGate','SoftGate','GatedResidual','GlobalResidual']
LAMBDAS=[0.0,0.2,0.5,1.0]
META_FOLDS=2 if FAST_MODE else 3
ALPHA=80.0
THRESHOLD_QUANTILE=.65

class RegimeExpert:
    def __init__(self,threshold_quantile=.65,alpha=80):
        self.q=threshold_quantile;self.alpha=alpha
    def fit(self,frame,x):
        y=frame.y_true.to_numpy(float);b=frame.base.to_numpy(float)
        self.threshold=float(np.quantile(y,self.q));self.count_high=int((y>=self.threshold).sum())
        target=(y>=self.threshold).astype(int)
        self.scale=StandardScaler().fit(x)
        xx=np.clip(self.scale.transform(x),-8,8)
        self.gate=LogisticRegression(C=.1,solver='liblinear',class_weight='balanced',max_iter=400,random_state=SEED)
        if min(np.bincount(target,minlength=2))<15:
            self.gate=None;self.pconst=float(target.mean())
        else:self.gate.fit(xx,target)
        self.abs_exp=[];self.res_exp=[]
        for mask in [target==0,target==1]:
            if mask.sum()<10: mask=np.ones(len(y),bool)
            ra=Ridge(alpha=self.alpha).fit(xx[mask],y[mask])
            rr=Ridge(alpha=self.alpha).fit(xx[mask],(y-b)[mask])
            self.abs_exp.append(ra);self.res_exp.append(rr)
        self.global_res=Ridge(alpha=self.alpha).fit(xx,y-b)
        return self
    def predict_components(self,frame,x):
        xx=np.clip(self.scale.transform(x),-8,8)
        p=self.gate.predict_proba(xx)[:,1] if self.gate is not None else np.full(len(xx),self.pconst)
        a0,a1=(m.predict(xx) for m in self.abs_exp)
        r0,r1=(m.predict(xx) for m in self.res_exp)
        b=frame.base.to_numpy(float)
        out={'HardGate':np.where(p>=.5,a1,a0),'SoftGate':(1-p)*a0+p*a1,
             'GatedResidual':b+np.clip((1-p)*r0+p*r1,-1.5,1.5),
             'GlobalResidual':b+np.clip(self.global_res.predict(xx),-1.5,1.5)}
        return out,p

def crossfit_regime(meta,features):
    unique=meta.scaffold_group.nunique();splits=min(META_FOLDS,unique)
    if splits<2:raise ValueError('Insufficient distinct scaffold groups for meta selection')
    raw=make_x(meta,features); pred={k:np.full(len(meta),np.nan) for k in METHODS}
    gate_oof=np.full(len(meta),np.nan);thresholds=[]
    for k,(tr,va) in enumerate(GroupKFold(n_splits=splits).split(raw,groups=meta.scaffold_group)):
        a=meta.iloc[tr];b=meta.iloc[va];guarded_folds(a,b)
        model=RegimeExpert(THRESHOLD_QUANTILE,ALPHA).fit(a,raw[tr]);est,prob=model.predict_components(b,raw[va]);
        for name in METHODS:pred[name][va]=est[name]
        gate_oof[va]=prob;thresholds.append(model.threshold)
    assert all(np.isfinite(x).all() for x in pred.values())
    selected={};b=meta.base.to_numpy(float);y=meta.y_true.to_numpy(float)
    for name in METHODS:
        scored=[]
        for lam in LAMBDAS:
            p=b+lam*(pred[name]-b)
            scored.append((float(np.mean(np.abs(p-y))),float(lam)))
        selected[name]=min(scored,key=lambda x:(x[0],x[1]))
    return selected,gate_oof,thresholds

archive_sig=hashlib.sha256(Path(NB11_RESULTS_ZIP).read_bytes()).hexdigest()[:12]
settings={'tag':EXPERIMENT_TAG,'archive':archive_sig,'fast':FAST_MODE,'bits':FP_BITS,
          'methods':METHODS,'lambdas':LAMBDAS,'alpha':ALPHA,'meta_folds':META_FOLDS,'threshold_quantile':THRESHOLD_QUANTILE}
sig=hashlib.sha256(json.dumps(settings,sort_keys=True).encode()).hexdigest()[:16]
CACHE=OUTPUT/'cache16'/sig;CACHE.mkdir(parents=True,exist_ok=True)
records=[];selections=[];gates=[];audits=[]

for fold in FOLDS_TO_RUN:
    cp=CACHE/f'fold{fold}_pred.csv';sp=CACHE/f'fold{fold}_selected.csv';gp=CACHE/f'fold{fold}_gate.csv';ap=CACHE/f'fold{fold}_audit.csv'
    if all(p.exists() for p in [cp,sp,gp,ap]):
        records.append(pd.read_csv(cp));selections.append(pd.read_csv(sp));gates.append(pd.read_csv(gp));audits.append(pd.read_csv(ap));print('Cache hit: fold',fold);continue
    rec=[];sel=[];gat=[];aud=[]
    train_all=INNER[INNER.fold==fold];held_all=OUTER[OUTER.fold==fold];guarded_folds(train_all,held_all)
    for iso in ISOS:
        tr=train_all[train_all.isoform==iso].reset_index(drop=True)
        te=held_all[held_all.isoform==iso].reset_index(drop=True)
        if min(len(tr),len(te))<25:raise ValueError(f'Too few {iso} labels: {len(tr)} training, {len(te)} held')
        guarded_folds(tr,te)
        xtr=make_x(tr,FEATURES);xte=make_x(te,FEATURES)
        choices,_,thresholds=crossfit_regime(tr,FEATURES)
        model=RegimeExpert(THRESHOLD_QUANTILE,ALPHA).fit(tr,xtr)
        raw,p=model.predict_components(te,xte)
        thr=model.threshold;yhigh=(te.y_true.to_numpy(float)>=thr).astype(int)
        gate_auc=safe_auc(yhigh,p)
        for i,r in te.reset_index(drop=True).iterrows():
            gat.append(dict(fold=fold,isoform=iso,canonical_smiles=r.canonical_smiles,scaffold_group=r.scaffold_group,
                 gate_probability=float(p[i]),true_high=int(yhigh[i]),gate_threshold=thr,
                 base=float(r.base),y_true=float(r.y_true)))
        for method in ['Baseline']+METHODS+['NestedBest']:
            if method=='Baseline':lam=0;predict=te.base.to_numpy(float);chosen='Baseline'
            elif method=='NestedBest':
                chosen=min(choices,key=lambda k:(choices[k][0],METHODS.index(k)))
                lam=choices[chosen][1];predict=te.base.to_numpy(float)+lam*(raw[chosen]-te.base.to_numpy(float))
            else:chosen=method;lam=choices[method][1];predict=te.base.to_numpy(float)+lam*(raw[method]-te.base.to_numpy(float))
            for i,r in te.reset_index(drop=True).iterrows():
                rec.append(dict(fold=fold,isoform=iso,method=method,canonical_smiles=r.canonical_smiles,
                    scaffold_group=r.scaffold_group,y_true=float(r.y_true),ci_low=float(r.ci_low),ci_high=float(r.ci_high),
                    base=float(r.base),y_pred=float(predict[i]),gate_probability=float(p[i]),gate_threshold=thr))
        for k,(score,lam) in choices.items():sel.append(dict(fold=fold,isoform=iso,method=k,inner_cv_mae=score,selected_strength=lam,selected=(k==min(choices,key=lambda v:(choices[v][0],METHODS.index(v))))))
        aud.append(dict(fold=fold,isoform=iso,n_meta_train=len(tr),n_outer=len(te),n_high_train=model.count_high,
                    gate_auc=gate_auc,gate_brier=float(np.mean((p-yhigh)**2)),
                    train_held_scaffold_overlap=len(set(tr.scaffold_group)&set(te.scaffold_group)),
                    mean_inner_threshold=float(np.mean(thresholds))))
    for data,path in [(rec,cp),(sel,sp),(gat,gp),(aud,ap)]:save_csv(pd.DataFrame(data),path)
    records.append(pd.read_csv(cp));selections.append(pd.read_csv(sp));gates.append(pd.read_csv(gp));audits.append(pd.read_csv(ap))
    print('Finished fold',fold)

pred=pd.concat(records,ignore_index=True);sel=pd.concat(selections,ignore_index=True)
gate=pd.concat(gates,ignore_index=True);audit=pd.concat(audits,ignore_index=True)
assert (audit.train_held_scaffold_overlap==0).all()
assert not pred.duplicated(['method','isoform','canonical_smiles']).any()
summary=[];per_iso=[];extremes=[]
for method,df in pred.groupby('method'):
    m=metric_stats(df,df.y_pred)
    summary.append({'method':method,**m})
    for iso,g in df.groupby('isoform'):
        per_iso.append({'method':method,'isoform':iso,'mae':float(np.mean(abs(g.y_true-g.y_pred))),
                    'interval_excess':float(np.mean(np.maximum(g.ci_low-g.y_pred,0)+np.maximum(g.y_pred-g.ci_high,0)))})
        qlow,qhi=np.quantile(g.y_true,[.1,.9]);den=g.y_true.std()
        for reg,mask in [('low_decile',g.y_true<=qlow),('high_decile',g.y_true>=qhi)]:
            part=g[mask]
            extremes.append({'method':method,'isoform':iso,'region':reg,'n':len(part),
                'mae':float(np.mean(abs(part.y_true-part.y_pred))),
                'signed_bias':float(np.mean(part.y_pred-part.y_true)),
                'spread_ratio':float(g.y_pred.std()/den) if den else np.nan})
summary=pd.DataFrame(summary).sort_values('macro_mae');per_iso=pd.DataFrame(per_iso);extremes=pd.DataFrame(extremes)
baseline=pred[pred.method=='Baseline'];candidate=pred[pred.method=='NestedBest']
boot=sample_bootstrap(candidate,B=200 if FAST_MODE else 1000)
for fname,df in [('outer_predictions.csv',pred),('method_macro_summary.csv',summary),('method_per_isoform.csv',per_iso),
                 ('potency_extremes.csv',extremes),('selected_meta_models.csv',sel),('gate_diagnostics.csv',gate),
                 ('leakage_audit.csv',audit),('paired_scaffold_bootstrap.csv',pd.DataFrame([boot]))]:save_csv(df,OUTPUT/fname)
metadata={'experiment':'Potency-regime MoE: hard/soft/gated residual','base_source':'Notebook11 real inner OOF + outer OOF',
    'selected_outer_folds':FOLDS_TO_RUN,'full_five_fold':set(FOLDS_TO_RUN)==set(range(5)),
    'meta_training_rule':'Only Notebook11 inner OOF rows from the outer-training molecules; scaffold CV selects strengths',
    'model_selection':'Inner meta-CV on MAE; outer holdout never consulted',
    'caveat':'Meta CV reuses inner-OOF first-stage predictions; inner base training can include other meta-validation rows. Outer test remains untouched. Scientific claims require independent reruns.',
    'archive_sha12':archive_sig,'config':settings,'scientifically_valid_source':bool(META11.get('scientifically_valid_full_model_run'))}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2))
RESULT_FILES=['outer_predictions.csv','method_macro_summary.csv','method_per_isoform.csv','potency_extremes.csv','selected_meta_models.csv','gate_diagnostics.csv','leakage_audit.csv','paired_scaffold_bootstrap.csv','run_metadata.json']
wrap_results(OUTPUT,'cyp_notebook_16_results.zip',RESULT_FILES)
print(summary.to_string(index=False));print('Paired scaffold bootstrap:',boot)


## 4. Visual comparison and research interpretation
Inspect the outer-fold tables before drawing conclusions. Improvements are exploratory unless repeated across seeds and split strategies. In particular, compare potent-decile error, interval-aware performance, gate/contrastive diagnostics, and robustness to removing each modeling component.

In [ ]:
import matplotlib.pyplot as plt
try:
    chart_df = summary.sort_values('macro_mae' if 'macro_mae' in summary else 'macro_st_rae')
    key = 'macro_st_rae' if 'macro_st_rae' in summary else 'macro_mae'
    fig,ax=plt.subplots(figsize=(9,4))
    ax.barh(chart_df.method[::-1],chart_df[key][::-1])
    ax.set_xlabel(key+' (lower is better)');ax.set_title('Nested outer-fold comparison')
    plt.tight_layout();plt.show()
except Exception as e:
    print('Optional figure unavailable:',type(e).__name__,str(e))


## 5. Reproducibility checklist
- [ ] Real Notebook 11 archive, no proxy checkpoint
- [ ] `FAST_MODE=False`, outer folds 0–4
- [ ] No held-out scaffold overlaps in audits
- [ ] No selecting thresholds/strengths from outer-fold outcomes
- [ ] Compare against the **fixed** 50:50 CheMeleon + screening baseline on exactly matched molecules
- [ ] Publish a negative result if the specialists fail to improve the strong baseline
- [ ] Record dataset versions, code revisions, and multiple seeds for further research